# 11 — Hyperparameter Tuning with Optuna
This notebook uses the **Optuna** framework to find the absolute best hyperparameters for our LightGBM model. This is how we bridge the gap to the top of the leaderboard.

**Objective:** Maximize Cross-Validation AUC.
**Search Space:** `learning_rate`, `num_leaves`, `max_depth`, `lambda_l1`, `lambda_l2`, `feature_fraction`.

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import pickle
from sklearn.metrics import roc_auc_score

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set project path
DRIVE_PATH = '/content/drive/MyDrive/santander-customer-satisfaction/'
PICKLE_DIR = DRIVE_PATH + 'pickles/'

# Install Optuna if not already installed
!pip install optuna
import optuna

print('Setup complete.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 28.6 MB/s eta 0:00:00
Setup complete.


## 1. Load Advanced Data & CV Folds

In [3]:
# USE THE ADVANCED FEATURES
train = pd.read_pickle(f'{PICKLE_DIR}train_advanced.pkl')
X = train.drop(columns=['TARGET', 'ID'], errors='ignore')
y = train['TARGET']

with open(f'{PICKLE_DIR}cv_fold_indices.pkl', 'rb') as f:
    cv_folds = pickle.load(f)

print(f"Loaded data with {X.shape[1]} features and 5-fold CV indices.")

Loaded data with 102 features and 5-fold CV indices.


## 2. Define Optuna Objective Function
This function will be called by Optuna for every trial. It trains a 5-fold CV and returns the mean AUC.

In [4]:
def objective(trial):
    # Define the search space
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 3000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
    }

    fold_aucs = []

    # Standard 5-fold CV loop using our fixed indices
    for fold_dict in cv_folds:
        train_idx, val_idx = fold_dict['train_idx'], fold_dict['val_idx']
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        dtrain = lgb.Dataset(X_train, label=y_train)
        dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

        model = lgb.train(
            param,
            dtrain,
            num_boost_round=2000,
            valid_sets=[dval],
            callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
        )

        preds = model.predict(X_val)
        fold_aucs.append(roc_auc_score(y_val, preds))

    return np.mean(fold_aucs)

## 3. Run Optuna Optimization
Change `n_trials` based on your available time. 50-100 trials are usually sufficient for a solid boost.

In [5]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50) # Run 50 trials

print("Best trial AUC:", study.best_value)
print("Best parameters:", study.best_params)

[I 2026-05-02 22:39:53,984] A new study created in memory with name: no-name-2c818b7b-d8b9-442b-a3c4-5896d9b22200
[I 2026-05-02 22:41:29,322] Trial 0 finished with value: 0.8394961618918785 and parameters: {'learning_rate': 0.0107996929219919, 'num_leaves': 319, 'max_depth': 3, 'feature_fraction': 0.5574151924742099, 'bagging_fraction': 0.8735731685388961, 'bagging_freq': 6, 'min_child_samples': 29, 'lambda_l1': 0.10140536923212182, 'lambda_l2': 0.00016991722419697512}. Best is trial 0 with value: 0.8394961618918785.
[I 2026-05-02 22:42:14,879] Trial 1 finished with value: 0.8379927332378448 and parameters: {'learning_rate': 0.012792329909457134, 'num_leaves': 383, 'max_depth': 10, 'feature_fraction': 0.6071189297223325, 'bagging_fraction': 0.7890461599980347, 'bagging_freq': 2, 'min_child_samples': 68, 'lambda_l1': 0.0004082625565403223, 'lambda_l2': 0.024436622582361216}. Best is trial 0 with value: 0.8394961618918785.
[I 2026-05-02 22:43:14,524] Trial 2 finished with value: 0.840286

Best trial AUC: 0.8412823423652303
Best parameters: {'learning_rate': 0.01016896744014576, 'num_leaves': 466, 'max_depth': 5, 'feature_fraction': 0.594613203819703, 'bagging_fraction': 0.6679353521265596, 'bagging_freq': 2, 'min_child_samples': 17, 'lambda_l1': 3.0459978968811314, 'lambda_l2': 3.147313080131433e-06}


## 4. Visualize Optimization Results

In [6]:
from optuna.visualization import plot_optimization_history, plot_param_importances

plot_optimization_history(study).show()
plot_param_importances(study).show()

## 5. Next Steps
1. Copy the **Best parameters** from above.
2. Paste them into a new version of `04_LightGBM.ipynb`.
3. Rerun the training to get optimized OOF/Test pickles!
4. (Optional) Repeat this process for XGBoost and CatBoost.